# 5.4 Trying AMCL in ROS2

After the previous notebook, we are now quite knowledgeable about how Monte Carlo Localization works and all the ideas that underpin it. However, the implementation we worked with is, of course, kind of a simple one. More production-ready versions of the Monte Carlo Localization algorithm include additional techniques, such as *adapting the number of particles* or *injecting random particles* when the robot gets lost.

One such implementation, very widely used in real robotics applications, is **[AMCL](https://docs.nav2.org/jazzy/configuration_and_development/configuration_guide/others/configuring_amcl/)** (which stands for *Adaptive* Monte Carlo Localization). Despite these extra techniques, at its core, AMCL is essentially the same method we are already familiar with. But, how well does it work? Well, luckily for us, a version of AMCL is included in **[Nav2](https://docs.nav2.org/)**, the navigation stack of ROS2, so we can quite easily test it on a simulated robot.

By the end of this notebook you will be able to run AMCL on a simulated robot, tell which data it uses and what it publishes, look at its particles and at the uncertainty of its estimate, explain how it gets lost and how it recovers, and see how many particles it needs. Optionally, you will also use its estimate to make the robot navigate on its own with Nav2.

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/localization/images/fig5-4-rviz.png" width="1100" alt="">
  <figcaption>Fig. 1. The AMCL demo of this notebook in RViz2. Left: the map of the warehouse, the laser scan, the pose estimated by AMCL (green arrow) and the pose given by the odometry alone (blue arrow), which has drifted away. Right: zoom on the robot at another moment, with the particles of AMCL and the uncertainty of its estimate, both in position (magenta ellipse) and in orientation (yellow cone).</figcaption>
</figure>

## 5.4.1 How to work with this notebook

**This notebook is not meant to be run.** ROS2 runs on your own machine, so this notebook plays the role of a **lab guide and report**: keep it open while you work on your terminals, and write your answers back into it:

1. **Commands**: paste the command you used (and, when asked, its output) into the markdown code blocks below.
2. **Screenshots**: a capture of RViz2 or of a terminal.

**Assumed setup.** Ubuntu 24.04, ROS2 Jazzy and MVSim (installed as a ROS2 package), plus **Nav2**, which includes AMCL, and `xterm`, a terminal emulator that the demo uses to open a window for driving the robot:

```bash
sudo apt install ros-jazzy-navigation2 ros-jazzy-nav2-bringup xterm
```

The demo lives in the `robotics_ws` folder of this repository, a ROS2 workspace with several packages that we have to build. From the root of the workspace:

```bash
cd robotics_ws
sudo rosdep init   # only the first time you use rosdep in your computer
rosdep update
rosdep install --from-paths src   # installs any dependency of the packages still missing
colcon build --symlink-install
```

Lastly, remember that you need to set up the workspace in every new terminal where you want to use its packages:

```bash
source install/setup.bash
```

*Tip: the robot is driven from a small `xterm` window with the arrow keys (each key press increases or decreases the linear or angular speed), the space bar stops it and `p` pauses the program. Click on that window before using the keys, and drive gently: if the robot crashes into something, its odometry, and maybe AMCL too, will get confused.*

## 5.4.2 The demo

The package `particle_filter_demo` of the workspace has a prepared example to test AMCL. It launches:

- **MVSim**, simulating a [Clearpath Jackal](https://clearpathrobotics.com/jackal-small-unmanned-ground-vehicle/) robot with a 2D laser scanner in a warehouse. We launch it in *headless* mode, that is, with no GUI of its own: we will watch everything in RViz2.
- **`odom_simulator`**, which computes the odometry of the robot by integrating the velocity commands sent to it. The Jackal is a skid-steered robot, so its wheels slip, especially when turning, and the robot does not move exactly as commanded. This odometry is quite accurate when driving straight, but drifts **a lot** when turning (or when the robot crashes into something), much more than a real one, which makes the effect of AMCL easy to see.
- **`map_server`**, a node of Nav2 that loads the map of the warehouse from a file and publishes it.
- **`amcl`**, the AMCL node of Nav2.
- **`keyboard_control`**, in an `xterm` window, to drive the robot with the arrow keys (the window must have the focus).
- **RViz2** (Fig. 2), with these displays (see also Fig. 1):
  - `Map` and `LaserScan`: the map and the current scan $\mathbf{z}_t$.
  - `pose_AMCL` and `particles_AMCL`: the pose estimated by AMCL (green arrow), its uncertainty (magenta ellipse and yellow cone) and its particles $\mathbf{x}_t^{[i]}$.
  - `pose_Odometry`: the pose given by the odometry alone (blue arrow).
  - `pose_GT`: the true pose of the robot, the *ground truth* (red arrow). It is disabled by default: a real robot does not know it!

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/localization/images/fig5-4-launched.png" width="1100" alt="">
  <figcaption>Fig. 2. The demo right after launching it: RViz2, with its <i>Displays</i> panel on the left, and the <code>xterm</code> window to drive the robot (bottom right).</figcaption>
</figure>

### **<span style="color:green"><b><i>ASSIGNMENT 1: Launching the demo</i></b></span>**

**What to do?**

1. Launch the demo (and wait a few seconds, the first time MVSim has to download some 3D models):

```bash
ros2 launch particle_filter_demo demo_warehouse.launch.py
```

   Drive the robot around for a while, and include a **screenshot** of RViz2.

```bash
# Terminal 1: dependencias, workspace y demo
sudo apt install ros-jazzy-navigation2 ros-jazzy-nav2-bringup xterm
cd ~/uma_robotics_2027/robotics_ws
rosdep update
rosdep install --from-paths src --ignore-src -y   # All required rosdeps installed successfully
colcon build --symlink-install                    # Summary: 4 packages finished
source install/setup.bash
ros2 launch particle_filter_demo demo_warehouse.launch.py
```

<img src="img/5.4_captura_assignment1_rviz.png" alt="RViz2 con el mapa del almacén, el scan, AMCL y la odometría tras conducir un rato" width="900">

<p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>La demo arranca sin errores: <code>map_server</code> carga el mapa del almacén (705 × 524 celdas de 0,05 m), el <code>lifecycle_manager</code> activa <code>map_server</code> y <code>amcl</code>, y AMCL fija la pose inicial en (0, 0, 0). En RViz2 se ven las estanterías del mapa, el robot con su nube de partículas verdes, la elipse magenta (incertidumbre en posición) y el cono amarillo (incertidumbre en orientación) de <code>pose_AMCL</code>. Tras conducir un rato, la flecha azul de <code>pose_Odometry</code> ya se ha quedado lejos del robot, encima de otra estantería: la odometría ha derivado, mientras que AMCL sigue al robot.</i></p>


2. From another terminal, list the running nodes. Then, find out the **inputs** and **outputs** of the main node of AMCL, that is, the topics it subscribes to and publishes (*hint: use a `ros2 node` subcommand*). In the table below, which only includes the topics related to localization, write whether each one is an input or an output, its message type and what it is. The first row is an example.

```bash
ros2 node list
# /amcl
# /ground_truth_tf
# /keyboard_control_plus
# /lifecycle_manager_navigation
# /map_server
# /mvsim
# /odom_sim
# /rviz2
# /transform_listener_impl_ab1e4a294ca0

ros2 node info /amcl
#   Subscribers:
#     /bond: bond/msg/Status
#     /initialpose: geometry_msgs/msg/PoseWithCovarianceStamped
#     /map: nav_msgs/msg/OccupancyGrid
#     /parameter_events: rcl_interfaces/msg/ParameterEvent
#     /scanner1: sensor_msgs/msg/LaserScan
#     /tf: tf2_msgs/msg/TFMessage
#     /tf_static: tf2_msgs/msg/TFMessage
#   Publishers:
#     /amcl/transition_event: lifecycle_msgs/msg/TransitionEvent
#     /amcl_pose: geometry_msgs/msg/PoseWithCovarianceStamped
#     /bond: bond/msg/Status
#     /parameter_events: rcl_interfaces/msg/ParameterEvent
#     /particle_cloud: nav2_msgs/msg/ParticleCloud
#     /rosout: rcl_interfaces/msg/Log
#     /tf: tf2_msgs/msg/TFMessage
#   Service Servers:
#     ...
#     /reinitialize_global_localization: std_srvs/srv/Empty
#     /request_nomotion_update: std_srvs/srv/Empty
#     /set_initial_pose: nav2_msgs/srv/SetInitialPose
```

| Input / Output | Topic | Message type | What is it? |
|---|---|---|---|
| Input | `/scanner1` | `sensor_msgs/msg/LaserScan` | The laser scan, that is, the observation $\mathbf{z}_t$ |
| Output | `/amcl_pose` | `geometry_msgs/msg/PoseWithCovarianceStamped` | La pose estimada por AMCL en el mapa, con su covarianza 6×6 (calculada a partir de las partículas) |
| Input | `/initialpose` | `geometry_msgs/msg/PoseWithCovarianceStamped` | La pose inicial (con su incertidumbre) que se le da a AMCL, p. ej. con *2D Pose Estimate* de RViz2: reinicia las partículas alrededor de ella |
| Input | `/map` | `nav_msgs/msg/OccupancyGrid` | El mapa de ocupación $\mathbf{m}$ que publica `map_server`; AMCL construye con él el *likelihood field* |
| Output | `/particle_cloud` | `nav2_msgs/msg/ParticleCloud` | Las partículas $\mathbf{x}_t^{[i]}$ con sus pesos $w_t^{[i]}$, es decir, la creencia $bel(\mathbf{x}_t)$ |
| Input | `/tf` | `tf2_msgs/msg/TFMessage` | Las transformaciones que lee: `noisy_odom → base_link` (la odometría, que usa como movimiento $\mathbf{u}_t$ en la predicción) y `base_link → scanner1` (dónde va montado el láser) |
| Output | `/tf` | `tf2_msgs/msg/TFMessage` | La transformación que publica: `map → noisy_odom`, la corrección de la deriva de la odometría |


3. Generate a diagram of the tf tree, as you did in notebook 4.3, and include a **screenshot** of it.

```bash
cd ~
ros2 run tf2_tools view_frames
# Genera frames_<fecha>.pdf con el árbol de los últimos 5 s de /tf y /tf_static
```

<img src="img/5.4_tf_tree.png" alt="Árbol tf de la demo de AMCL generado con view_frames" width="800">

<p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>El árbol es <b><code>map</code> &rarr; <code>noisy_odom</code> &rarr; <code>base_link</code> &rarr; {<code>scanner1</code>, <code>base_footprint</code>}</b>, más una rama <code>map</code> &rarr; <code>odom</code> (estática e identidad, publicada por <code>ground_truth_tf</code>, solo para dibujar la pose real). Las frecuencias encajan con quién publica cada una: <code>map</code> &rarr; <code>noisy_odom</code> sale a ~9,9 Hz, el ritmo de los scans con los que AMCL corrige; <code>noisy_odom</code> &rarr; <code>base_link</code>, a ~39,7 Hz, el ritmo de la odometría; <code>base_link</code> &rarr; <code>scanner1</code>, a ~10 Hz, y <code>base_footprint</code> es fija (por eso <code>view_frames</code> marca 10000 Hz y tiempo 0).</i></p>


4. In the tree, `map` → `noisy_odom` is published by AMCL, and `noisy_odom` → `base_link` by `odom_simulator`, the node that simulates the odometry. *Note: the frame named `odom` is only used to show the ground truth in RViz2, ignore it.*
   - AMCL estimates the pose of the robot in the map, `map` → `base_link`. Why can't it publish it directly in tf? *Hint: how many parents can a frame have?*
   - What does `map` → `noisy_odom` represent, then? How do we get the pose estimated by AMCL from it?

<p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i><b>¿Por qué AMCL no publica <code>map</code> &rarr; <code>base_link</code>?</b> Porque tf es un árbol: cada frame tiene <b>un solo padre</b>. <code>base_link</code> ya tiene padre, <code>noisy_odom</code>, que publica <code>odom_simulator</code>. Si AMCL publicara también <code>map</code> &rarr; <code>base_link</code>, <code>base_link</code> tendría dos padres y el árbol dejaría de ser un árbol.<br>
<b>¿Qué representa <code>map</code> &rarr; <code>noisy_odom</code>?</b> La <b>corrección de la deriva de la odometría</b>: dónde está en el mapa el origen desde el que la odometría cuenta. AMCL la calcula para que, compuesta con la odometría, dé su estimación:<br>
&nbsp;&nbsp;<code>x_AMCL = (map &rarr; base_link) = (map &rarr; noisy_odom) &oplus; (noisy_odom &rarr; base_link)</code>, es decir, <code>(map &rarr; noisy_odom) = x_AMCL &oplus; (x_odom)<sup>&minus;1</sup></code>.<br>
Si la odometría fuera perfecta, esta transformación se quedaría en la identidad (el robot arranca en (0, 0, 0)); a medida que la odometría deriva, crece lo que hay que corregir. Para obtener la pose estimada por AMCL basta con pedir a tf <code>map</code> &rarr; <code>base_link</code> (p. ej. <code>ros2 run tf2_ros tf2_echo map base_link</code>), que compone las dos transformaciones, o leerla directamente en <code>/amcl_pose</code>. Así se combinan la odometría, suave y rápida (~40 Hz), con la corrección de AMCL, más lenta (~10 Hz) pero sin deriva.</i></p>

## 5.4.3 Odometry vs. AMCL

The demo gives us two estimates of the pose of the robot:

- The **odometry** (`pose_Odometry`, in blue) just integrates the motion of the robot. In terms of the Bayes filter, it only does the **prediction** step, over and over, so its error keeps growing.
- **AMCL** (`pose_AMCL`, in green) does the prediction too, but also the **correction** with each scan, comparing it with the map through the likelihood field, as we did in notebook 5.3.

Let's see how well each of them does, comparing them with the **ground truth** (`pose_GT`, in red), the real pose of the robot, which the simulator knows (Figs. 3 and 4).

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/localization/images/fig5-4-ground-truth.png" width="1100" alt="">
  <figcaption>Fig. 3. The three poses of the robot in RViz2 right after launching the demo, once <code>pose_GT</code> has been ticked in the <i>Displays</i> panel: they are still together.</figcaption>
</figure>

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/localization/images/fig5-4-ground-truth-top.png" width="1100" alt="">
  <figcaption>Fig. 4. The same moment as in Fig. 3, seen from above: the three poses coincide.</figcaption>
</figure>

### **<span style="color:green"><b><i>ASSIGNMENT 2: Who knows where the robot is?</i></b></span>**

**What to do?**

1. Enable the `pose_GT` display of RViz2, by ticking its box in the *Displays* panel. Drive the robot for a couple of minutes, turning a lot (for instance, around several rows of shelves, or spinning in place a few times), and include a **screenshot** showing the three poses: the ground truth, the odometry and the AMCL estimate.

<img src="img/5.4_captura_assignment2_tres_poses.png" alt="Las tres poses en RViz2: real (roja), AMCL (verde) y odometría (azul)" width="900">

<p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>Tras conducir girando varias veces, la <b>flecha azul</b> (odometría) se ha quedado lejos del robot y apunta en otra dirección, mientras que la <b>verde</b> (AMCL) sigue encima de la <b>roja</b> (pose real), con las partículas concentradas alrededor y los puntos del láser cayendo sobre las estanterías del mapa.<br>
<b>Nota:</b> en un primer intento AMCL acabó perdido (a 13 m de la pose real, con &sigma;<sub>&theta;</sub> &asymp; 50° y el scan sin encajar con el mapa) tras conducir demasiado rápido: había convergido a otro pasillo casi igual del almacén. Se relanzó la demo y se condujo despacio; las capturas y los datos de esta entrega son del segundo intento.</i></p>


2. Stop the robot (space bar) and print the three poses: the ground truth (`/base_pose_ground_truth`), the odometry (`/noisy_odom`) and the AMCL estimate (`/amcl_pose`). Fill in the table below with their positions and the error of each estimate with respect to the ground truth. *Hint: `--field pose.pose` prints only the pose of a message.*

```bash
ros2 topic echo --once /base_pose_ground_truth --field pose.pose
# position: {x: 2.9444894790649414, y: -0.17480693757534027, z: 0.0}
# orientation: {z: -0.7458718728593564, w: 0.666089445403}          -> yaw = -96.5°

ros2 topic echo --once /noisy_odom --field pose.pose
# position: {x: 1.632534900944122, y: -3.544933486307292, z: 0.0}
# orientation: {z: -0.6063247420083988, w: 0.7952171447022844}      -> yaw = -74.6°

ros2 topic echo --once /amcl_pose --field pose.pose
# pose: position: {x: 2.988199517605907, y: -0.06404700310062375, z: 0.0}
#       orientation: {z: -0.7394309552585138, w: 0.6732323985114514} -> yaw = -95.4°
```

| | $x$ (m) | $y$ (m) | Position error (m) |
|---|---|---|---|
| Ground truth | 2.944 | −0.175 | — |
| Odometry | 1.633 | −3.545 | 3.616 |
| AMCL | 2.988 | −0.064 | 0.119 |

<p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>Error de posición = distancia euclídea a la pose real, p. ej. odometría: &radic;((1.633 &minus; 2.944)&sup2; + (&minus;3.545 + 0.175)&sup2;) = &radic;(1.719 + 11.357) = 3.616 m. En orientación (yaw = 2&middot;atan2(z, w)), la odometría se equivoca en 21.8° y AMCL en 1.1°.</i></p>


3. Why does the error of the odometry keep growing, while the error of AMCL does not?

<p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>La <b>odometría</b> solo hace el paso de <b>predicción</b> del filtro de Bayes, una y otra vez: integra los incrementos de movimiento medidos, cada uno con su pequeño error, y no tiene ninguna referencia externa con la que corregirlos. Los errores se acumulan, y los de <b>orientación</b> son los peores: un grado mal estimado en un giro desvía todos los desplazamientos posteriores, así que el error en posición crece cada vez más deprisa (en esta demo, sobre todo al girar, porque el Jackal patina). Es <i>dead reckoning</i>: el error no está acotado.<br>
<b>AMCL</b> también predice con la odometría, pero en cada actualización <b>corrige</b>: compara el scan con el mapa a través del <i>likelihood field</i>, da poco peso a las partículas cuyo scan no encaja y el remuestreo las elimina. Como el mapa es fijo y no deriva, el error queda <b>acotado</b> por la precisión del láser y del mapa (aquí, ~0.1 m), da igual cuánto se conduzca. Eso sí, solo mientras siga la hipótesis correcta: el primer intento mostró que, si converge a un sitio equivocado que se parece al real, el error puede ser enorme.</i></p>

## 5.4.4 The particles

AMCL publishes its particles in the `/particle_cloud` topic, whose messages are of type `nav2_msgs/msg/ParticleCloud`. In RViz2, the `particles_AMCL` display draws each particle as a green arrow (Fig. 5): if they are hard to see, make them longer with its *Min Arrow Length* and *Max Arrow Length* options. Two details are worth knowing:

- AMCL does not update the filter with every scan, but only when the robot has moved more than 5 cm or turned more than 0.05 rad, about 3° (values set in the configuration of this demo, see section 5.4.7). So, **the particles are only published while the robot moves**: keep it moving when you print them.
- The *Adaptive* of AMCL: the number of particles $M$ is not fixed, but it is adapted after each update, between 500 and 5000 (again, values set in the configuration of this demo), using **KLD-sampling**. Roughly speaking, it uses few particles when they are concentrated in a small region, and many when they are spread out, so the computational effort follows the uncertainty.

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/localization/images/fig5-4-particles.png" width="1100" alt="">
  <figcaption>Fig. 5. The particles of AMCL around the robot in RViz2, with their arrows made longer in the <code>particles_AMCL</code> display.</figcaption>
</figure>

### **<span style="color:green"><b><i>ASSIGNMENT 3: Looking at the particles</i></b></span>**

**What to do?**

1. While the robot moves, print a single message of `/particle_cloud` without its long arrays (*hint: `ros2 topic echo` has a `--no-arr` option, which prints only the length of the arrays*). How many particles $M$ is AMCL using? Why that number?

```bash
# Con el robot girando despacio (las partículas solo se publican mientras se mueve)
ros2 topic echo --once --no-arr /particle_cloud
# header:
#   stamp: {sec: 1795286682, nanosec: 594339005}
#   frame_id: map
# particles: '<sequence type: nav2_msgs/msg/Particle, length: 501>'
```

<p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>AMCL usa <b>M = 501 partículas</b>, prácticamente el mínimo configurado (<code>min_particles: 500</code>). Con <b>KLD-sampling</b>, M se elige en cada remuestreo para que el error de aproximar la creencia con partículas quede acotado: el número necesario crece con cuántas celdas del espacio de poses ocupan las partículas. Como AMCL está bien localizado, todas caen en una zona de pocos centímetros y pocos grados (ver Assignment 4), el límite KLD sale por debajo de 500 y se aplica el mínimo. La partícula de más se debe a cómo está hecho el bucle de remuestreo de AMCL: añade partículas hasta <i>superar</i> el límite, así que se para en 500 + 1.<br>
Como contraste, en el primer intento, con AMCL perdido y las partículas dispersas, el mismo comando dio <b>687</b>: más dispersión, más partículas. Ese es justo el «Adaptive» de AMCL: el esfuerzo de cálculo sigue a la incertidumbre.</i></p>


2. Now print the particles themselves, and look at their weights. Are they different? Why do you think this happens? *Hint: the message is very long, so pipe the output of `ros2 topic echo` into `head -c 2000` to see only its first 2000 characters.*

```bash
ros2 topic echo --once /particle_cloud | head -c 2000
# particles:
# - pose:
#     position: {x: 3.051432394601989, y: -0.07939298570221588, z: 0.0}
#     orientation: {z: -0.4520876302659969, w: 0.8919735279482656}
#   weight: 0.001996007984031936
# - pose:
#     position: {x: 3.0309559475400514, y: -0.07667754492936985, z: 0.0}
#     orientation: {z: -0.4169170938790818, w: 0.9089445180160453}
#   weight: 0.001996007984031936
# - pose:
#     position: {x: 2.9753261436656824, y: -0.0544412338413996, z: 0.0}
#     orientation: {z: -0.40902882659456213, w: 0.9125214622213965}
#   weight: 0.001996007984031936
# - pose:
#     position: {x: 2.876225573425823, y: -0.04115057192941468, z: 0.0}
#     orientation: {z: 0.3835701543885716, w: -0.9235117414859041}
#   weight: 0.001996007984031936
# ...  (todas con weight: 0.001996007984031936)
```

<p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i><b>No, todos los pesos son iguales:</b> 0.001996007984031936 = 1/501 = 1/M. Ocurre porque AMCL publica las partículas <b>justo después del remuestreo</b> (con <code>resample_interval: 1</code> remuestrea en cada actualización), y tras remuestrear todas las partículas pasan a pesar lo mismo, 1/M, como en el <code>resample()</code> del notebook 5.3: la información de los pesos de la corrección ya está codificada en <b>cuántas copias</b> de cada pose hay, no en su peso.<br>
También se ve que las partículas están muy juntas: x entre 2.83 y 3.05 m, y entre &minus;0.09 y 0.00 m, y yaw entre &minus;55° y &minus;42°, coherente con &sigma; &asymp; 0.1 m y unos grados. Algunos cuaterniones tienen z &gt; 0 y w &lt; 0 (p. ej. z = 0.384, w = &minus;0.924), pero representan el mismo giro que su opuesto (&minus;q &equiv; q): su yaw es &minus;45°, como el de las demás.</i></p>

## 5.4.5 The uncertainty of the estimate

AMCL publishes its estimate in `/amcl_pose`: the pose, plus its **covariance**, which AMCL computes from its particles, as `current_estimation()` did in notebook 5.3. RViz2 draws it as the magenta ellipse (position) and the yellow cone (orientation) of `pose_AMCL`, both clearly visible in Fig. 5.

Since ROS represents poses in 3D, this covariance is a $6 \times 6$ matrix over $(x, y, z, \text{roll}, \text{pitch}, \text{yaw})$, stored as a list of 36 numbers, row by row. For our robot, the interesting ones are the variances of $x$, $y$ and the yaw $\theta$: the elements 0 ($\sigma_x^2$), 7 ($\sigma_y^2$) and 35 ($\sigma_\theta^2$) of the list.

### **<span style="color:green"><b><i>ASSIGNMENT 4: Reading the covariance</i></b></span>**

**What to do?**

1. Print a single message of `/amcl_pose`, and compute the standard deviations $\sigma_x$, $\sigma_y$ (in meters) and $\sigma_\theta$ (in degrees). Do they match the size of the ellipse and the cone in RViz2?

```bash
ros2 topic echo --once /amcl_pose
# header: {frame_id: map}
# pose:
#   pose: position: {x: 2.988199517605907, y: -0.06404700310062375}, orientation: {z: -0.7394309552585138, w: 0.6732323985114514}
#   covariance:          (36 valores, fila a fila, sobre x, y, z, roll, pitch, yaw)
#   [0]  = 0.00553823284647151      <- var(x)
#   [1]  = 0.00012326240315165915   <- cov(x, y)
#   [6]  = 0.00012326240315165915   <- cov(y, x)
#   [7]  = 0.012989778713032116     <- var(y)
#   [35] = 0.011816560162750919     <- var(yaw)
#   (el resto, 0.0)
```

<img src="img/5.4_captura_assignment4_elipse.png" alt="Zoom sobre el robot: elipse magenta pequeña y cono amarillo estrecho de pose_AMCL" width="900">

<p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i><b>Desviaciones típicas:</b><br>
&bull; &sigma;<sub>x</sub> = &radic;0.005538 = <b>0.074 m</b> (7.4 cm)<br>
&bull; &sigma;<sub>y</sub> = &radic;0.012990 = <b>0.114 m</b> (11.4 cm)<br>
&bull; &sigma;<sub>&theta;</sub> = &radic;0.011817 = 0.109 rad = <b>6.2°</b><br>
La covarianza x–y es casi nula (correlación &rho; = 0.000123 / (0.074 &middot; 0.114) &asymp; 0.015), así que los ejes de la elipse van alineados con x e y, y como &sigma;<sub>x</sub> y &sigma;<sub>y</sub> son del mismo orden, se ve <b>casi como un círculo</b>.<br>
<b>¿Encaja con RViz2?</b> Sí: la elipse magenta mide del orden de una o dos décimas de metro, más pequeña que el propio robot (el Jackal mide ~0.5 m), y por eso queda casi escondida debajo de él; el cono amarillo es estrecho, de unos pocos grados a cada lado de la flecha, coherente con &sigma;<sub>&theta;</sub> &asymp; 6°. Además, la incertidumbre es realista: el error real era de 0.12 m en posición (&asymp; 1&sigma;) y 1.1° en orientación.<br>
<b>Comparación con el intento en que AMCL se perdió:</b> allí &sigma;<sub>x</sub> = 0.77 m, &sigma;<sub>y</sub> = 0.12 m y &sigma;<sub>&theta;</sub> = 49.5°, con una elipse grande y alargada en x (la dirección del pasillo, donde el scan apenas fija cuánto se ha avanzado) y un cono amarillo muy abierto. Las partículas estaban dispersas y la covarianza lo reflejaba, aunque la media estuviera a 13 m de la pose real.</i></p>

## 5.4.6 Global localization and the kidnapped robot

So far, AMCL has been solving the easiest localization problem of notebook 5.0, **position tracking**: it starts knowing the initial pose of the robot, $[0, 0, 0]^T$, where the robot starts. But, unlike the EKF, a particle filter can represent multimodal beliefs, so it can also tackle the two harder ones:

- **Global localization**: the initial pose is unknown. AMCL spreads the particles over the free space of the map, and lets the scans discard the wrong hypotheses (Fig. 6).
- **The kidnapped robot**: the robot is moved somewhere else without the filter knowing it, so all the particles are wrong. To recover, when the scans suddenly fit the map much worse than usual, AMCL **injects random particles** all over the map. This is the *Augmented MCL* algorithm of the book *Probabilistic Robotics*, by S. Thrun, W. Burgard and D. Fox.

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/localization/images/fig5-4-global-localization.png" width="1100" alt="">
  <figcaption>Fig. 6. Global localization with AMCL in the warehouse of the demo, from the particles published in <code>/particle_cloud</code> while the robot moved. (a) Right after the reinitialization, the M = 5000 particles cover the whole map. (b) A few updates later, only those where the scan fits the map survive, grouped in clusters. (c) A few hypotheses remain, and their number drops. (d) The particles converge into a single cluster, at the true pose of the robot (in red).</figcaption>
</figure>

### **<span style="color:green"><b><i>ASSIGNMENT 5: Getting lost, and found</i></b></span>**

**What to do?**

1. **Global localization**. AMCL offers a service, `/reinitialize_global_localization`, that spreads all its particles over the map. Find out its type, and call it. *Hint: `ros2 service type` and `ros2 service call`.* Then, drive the robot slowly, avoiding the obstacles, until the particles converge, and include **screenshots** of RViz2 right after starting to drive and once they have converged. Did it converge to the right place?

```bash
ros2 service type /reinitialize_global_localization
# std_srvs/srv/Empty

# Con el robot parado: reinicializar y esperar al primer mensaje de partículas (llega en cuanto el robot se mueve)
ros2 service call /reinitialize_global_localization std_srvs/srv/Empty && ros2 topic echo --once --no-arr /particle_cloud
# requester: making request: std_srvs.srv.Empty_Request()
# response: std_srvs.srv.Empty_Response()
# particles: '<sequence type: nav2_msgs/msg/Particle, length: 5000>'

# Tras converger, con el robot parado: pose real frente a pose de AMCL
ros2 topic echo --once /base_pose_ground_truth --field pose.pose.position
# x: 18.672639846801758   y: 9.316012382507324
ros2 topic echo --once /amcl_pose --field pose.pose.position
# x: -2.544240145380051   y: 6.650917938888743
```

<img src="img/5.4_captura_assignment5_global_inicio.png" alt="Justo tras reinicializar: las 5000 partículas repartidas por todo el almacén" width="900">

<img src="img/5.4_captura_assignment5_global_convergido.png" alt="Tras conducir: las partículas convergen en un solo grupo" width="900">

<p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>El servicio es de tipo <code>std_srvs/srv/Empty</code>: no lleva datos ni en la petición ni en la respuesta, solo dispara la acción. Al llamarlo, AMCL reparte las partículas de manera uniforme por el espacio libre del mapa con el <b>máximo de partículas, M = 5000</b> (KLD-sampling necesita muchas para cubrir una creencia tan extendida). En la primera captura se ven en grupos por todos los pasillos, con la elipse magenta ocupando medio almacén y el cono amarillo cruzándolo entero: con la creencia repartida, la media ponderada no significa nada.<br>
Al conducir, cada scan da poco peso a las partículas donde no encaja con el mapa, el remuestreo las elimina y M baja; en pocas actualizaciones queda <b>un único grupo</b> (segunda captura).<br>
<b>¿Convergió al sitio correcto? No.</b> La pose real era (18.67, 9.32) y AMCL convergió a (&minus;2.54, 6.65): a <b>21.4 m</b>, en otra zona del almacén. Lo que ocurre es que el almacén tiene muchos sitios casi idénticos (filas de estanterías iguales y paralelas), así que, para un láser 2D, un pasillo y otro producen scans casi iguales y la verosimilitud no los distingue. El filtro elige pronto una de las hipótesis compatibles y descarta las demás, entre ellas la correcta: con 5000 partículas en todo el almacén hay pocas cerca de la pose real con la orientación adecuada, y basta un par de remuestreos para perderlas. Un primer intento acabó igual, con las partículas fusionadas en el sitio equivocado en uno o dos ciclos. Es justo lo que advierte el enunciado del OPTIONAL: la localización global falla a menudo en este entorno.</i></p>


2. **The kidnapped robot**. With AMCL well localized, use the *2D Pose Estimate* tool of the toolbar of RViz2 (Fig. 7) to tell AMCL that the robot is somewhere else, e.g. in another aisle. Then, drive the robot, and include **screenshots** of RViz2 right after the kidnapping and a few seconds later. Does AMCL recover? How long does it take?

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/localization/images/fig5-4-rviz-tools.png" width="1100" alt="">
  <figcaption>Fig. 7. The tools of the toolbar of RViz2 used in this notebook: <i>2D Pose Estimate</i>, to give AMCL the pose of the robot, and <i>2D Goal Pose</i>, to send a navigation goal to Nav2.</figcaption>
</figure>

```bash
# Secuestro con la herramienta 2D Pose Estimate de RViz2 (publica en /initialpose,
# con la covarianza de Tool Properties: 0.25 m² en x e y, 0.0685 rad² en yaw).
# Tras conducir un rato, con el robot parado:
ros2 topic echo --once /base_pose_ground_truth --field pose.pose.position
# x: 18.305702209472656   y: 6.002810478210449
ros2 topic echo --once /amcl_pose --field pose.pose.position
# x: -1.8597400598060188  y: -6.27496504402811
```

<img src="img/5.4_captura_assignment5_secuestro_inicio.png" alt="Justo después del secuestro: las partículas alrededor de la pose falsa" width="900">

<img src="img/5.4_captura_assignment5_secuestro_despues.png" alt="Unos segundos después: AMCL concentrado en un pasillo que no es el real" width="900">

<p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>Justo después del secuestro (primera captura), AMCL coloca todas sus partículas en una nube gaussiana alrededor de la pose que le hemos dado, con el tamaño de la covarianza de la herramienta (&sigma; = 0.5 m en x e y, &asymp; 15° en yaw), y el scan no encaja con el mapa a su alrededor.<br>
<b>¿Se recupera? En esta prueba, no.</b> Tras conducir un rato, la nube se había concentrado de nuevo (segunda captura), con los puntos del láser alineados con el borde de una estantería, pero en un pasillo que no es el real: la pose real era (18.31, 6.00) y AMCL estaba en (&minus;1.86, &minus;6.27), a <b>23.6 m</b>. No se cronometró, porque no llegó a recuperarse mientras se condujo.<br>
<b>Por qué.</b> AMCL se recupera inyectando partículas aleatorias por el mapa (<i>Augmented MCL</i>): compara la media a corto plazo de la verosimilitud de los scans (<code>recovery_alpha_fast: 0.2</code>) con la media a largo plazo (<code>recovery_alpha_slow: 0.001</code>) e inyecta con probabilidad max(0, 1 &minus; w<sub>fast</sub>/w<sub>slow</sub>). Eso solo funciona si los scans pasan a encajar <b>mucho peor de lo habitual</b>. En este almacén, el robot secuestrado ve un pasillo de estanterías muy parecido al que AMCL cree tener delante, así que la verosimilitud apenas baja, w<sub>fast</sub> no cae por debajo de w<sub>slow</sub> y casi no se inyectan partículas. Y las pocas que se inyectan tienen que caer, por azar, cerca de la pose real <i>y</i> con la orientación adecuada para sobrevivir al remuestreo. Es el mismo problema de los sitios que se parecen que hizo fallar la localización global.<br>
<b>Nota:</b> la prueba se hizo partiendo de AMCL ya perdido tras la localización global (en otro pasillo), no bien localizado, así que mide sobre todo la recuperación desde una pose errónea. Además, en esta demo el nodo <code>odom_simulator</code> también está suscrito a <code>/initialpose</code> y, al recibir la pose falsa, reinicia la odometría simulada a esa pose.</i></p>

## 5.4.7 How many particles?

AMCL has quite a few parameters, all of them in `robotics_ws/src/particle_filter_demo/resources/params.yaml` (the [Nav2 documentation](https://docs.nav2.org/jazzy/configuration_and_development/configuration_guide/others/configuring_amcl/) describes them). We will play with two of them: `min_particles` and `max_particles` (500 and 5000), the limits of the number of particles $M$ for KLD-sampling (section 5.4.4).

You can change a parameter with `ros2 param set` while AMCL runs. Note that, when the number of particles changes, AMCL restarts the filter at the initial pose: give it the right pose with *2D Pose Estimate* afterwards.

### **<span style="color:green"><b><i>ASSIGNMENT 6: How many particles do we need?</i></b></span>**

**What to do?**

1. Reduce the number of particles a lot, setting `min_particles` to 10 and `max_particles` to 20, and give AMCL the right pose. Drive the robot for a while, turning several times, and watch `pose_AMCL` and `pose_GT`. Is the localization better or worse than with the default values? Does the ellipse reflect the actual error?

```bash
# Demo recién relanzada (robot y AMCL en (0, 0, 0)), antes de mover el robot
ros2 param set /amcl min_particles 10
ros2 param set /amcl max_particles 20

# Tras conducir un rato girando varias veces, con el robot parado
ros2 topic echo --once /base_pose_ground_truth --field pose.pose.position
# x: 2.3087573051452637   y: -0.5088871717453003
ros2 topic echo --once /amcl_pose
# position: {x: 1.5318700630692283, y: -0.6954262365289542}
# orientation: {z: 0.48292773885263607, w: 0.8756602075272578}
# covariance[0]  = 0.04908168732281978    <- var(x)
# covariance[1]  = 0.021265669771932526   <- cov(x, y)
# covariance[7]  = 0.011059623906085574   <- var(y)
# covariance[35] = 0.01538769062097332    <- var(yaw)
```

<img src="img/5.4_captura_assignment6_pocas_particulas.png" alt="AMCL con como mucho 20 partículas: la flecha verde se ha separado de la roja" width="900">

<p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i><b>Peor que con los valores por defecto.</b> Con como mucho 20 partículas, después de unos cuantos giros el error de posición es de &radic;(0.777&sup2; + 0.187&sup2;) = <b>0.80 m</b>, frente a los 0.12 m del Assignment 2 con 500–5000 partículas, y en la captura se ve que la flecha verde (AMCL) no solo está desplazada respecto a la roja (pose real), sino que también apunta en otra dirección. Las pocas partículas (flechitas verdes sueltas alrededor del robot) no bastan para cubrir la incertidumbre que añade la odometría en cada giro: si ninguna cae cerca de la pose real, el remuestreo no puede quedarse con ella, y como tras cada remuestreo muchas son copias de las mismas, la diversidad se pierde enseguida (<i>empobrecimiento de partículas</i>).<br>
<b>¿Refleja la elipse el error real? No, AMCL es demasiado optimista.</b> De la covarianza: &sigma;<sub>x</sub> = &radic;0.0491 = 0.22 m, &sigma;<sub>y</sub> = &radic;0.0111 = 0.105 m, &sigma;<sub>&theta;</sub> = &radic;0.0154 = 0.124 rad = 7.1°, y una correlación x–y de 0.021 / (0.22 &middot; 0.105) = 0.91, así que la elipse es estrecha y alargada en diagonal (semiejes de 0.24 m y 0.04 m). El error real, 0.78 m en x, es unas 3.5 &sigma;<sub>x</sub>; teniendo en cuenta la forma de la elipse, la distancia de Mahalanobis es &asymp; 4.9, muy fuera de la elipse de 3&sigma;. Lo mismo con el cono: &sigma;<sub>&theta;</sub> &asymp; 7° es estrecho, y la orientación está claramente más equivocada. La covarianza se calcula a partir de las partículas, y con tan pocas, y muchas repetidas, su dispersión es pequeña aunque estén todas en el sitio equivocado: el filtro cree saber dónde está mejor de lo que realmente lo sabe.</i></p>

### <font color="blue"><b><i>Thinking about it</i></b></font>

Having played with AMCL, **answer the following questions**:

- Right after starting, the odometry may be as accurate as AMCL, or even more. Why is AMCL needed, then?

  <p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>Porque el error de la odometría <b>crece sin límite</b>: al principio es pequeño, pero cada giro y cada patinazo se acumulan y no hay nada que lo corrija (en el Assignment 2, 3.6 m tras un par de minutos). AMCL, en cambio, corrige cada pocos centímetros con el láser y el mapa, así que su error queda <b>acotado</b> (~0.1 m) por mucho que se conduzca. Además, AMCL da la pose en el marco del <b>mapa</b> (la odometría solo es relativa al punto de partida), da una estimación de su incertidumbre (la covarianza) y, a diferencia de la odometría, puede abordar la localización global y el robot secuestrado, aunque en este almacén le cueste.</i></p>

- The odometry of this demo drifts a lot when turning, and yet AMCL keeps the robot well localized. Would it work equally well with no odometry at all? And with no laser scanner?

  <p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i><b>Sin odometría, no igual de bien.</b> La odometría es el <b>movimiento u<sub>t</sub> de la predicción</b>: aunque derive al girar, entre dos actualizaciones (cada 5 cm o 3°) su error es pequeño y lleva las partículas casi al sitio correcto, y el láser solo tiene que corregir lo poco que falta (por eso la configuración pone <code>alpha1</code> alto, para que las partículas se abran lo suficiente en los giros). Sin ella habría que suponer que el robot puede haberse movido en cualquier dirección y abrir mucho la nube en cada paso: harían falta muchas más partículas, sería más lento y se perdería con facilidad al moverse rápido o en pasillos que se parecen. Además, el AMCL de Nav2 necesita la tf <code>noisy_odom</code> &rarr; <code>base_link</code> para funcionar.<br>
<b>Sin láser, sería solo odometría.</b> Sin observaciones no hay <b>corrección</b>: los pesos no cambian, las partículas solo se mueven con el modelo de movimiento y la nube se abre más y más. La media seguiría a la odometría, con su misma deriva, y la covarianza crecería sin parar. El láser es lo que ata la estimación al mapa.</i></p>

- The warehouse of the demo has many places that look alike. What could you do to make global localization more reliable in such an environment?

  <p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>&bull; <b>Romper la simetría del entorno</b>: añadir landmarks o balizas distinguibles (reflectores, códigos QR o AprilTags en las estanterías, balizas UWB o WiFi) y usarlas en el modelo de observación.<br>
&bull; <b>Usar más información de los sensores</b>: más haces por scan (<code>max_beams</code> es 100), un láser 3D o una cámara que vean lo que hay en las estanterías, no solo su contorno.<br>
&bull; <b>Mantener varias hipótesis más tiempo</b>: más partículas (el OPTIONAL siguiente), o remuestrear con menos frecuencia (<code>resample_interval</code>), para no descartar demasiado pronto la hipótesis correcta.<br>
&bull; <b>Moverse con intención</b> (localización activa): llevar el robot hacia una zona distintiva (el final de un pasillo, un cruce, una zona con objetos únicos) antes de fiarse de la estimación.<br>
&bull; <b>No partir de cero</b>: dar una pose inicial aproximada con <i>2D Pose Estimate</i> o guardar la última pose conocida (AMCL ya guarda la pose cada 2 s con <code>save_pose_rate: 0.5</code>) y reutilizarla al arrancar.</i></p>

### **<span style="color:green"><b><i>OPTIONAL: More particles for global localization</i></b></span>**

In Assignment 5, global localization often converged to a wrong place. Set `max_particles` to 20000 and `min_particles` back to 500, and repeat it a few times. Does it converge to the right place more often? What is the price to pay?

```bash
# TODO: your command here
```

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

### **<span style="color:green"><b><i>OPTIONAL: Navigating with Nav2</i></b></span>**

Knowing where the robot is, is usually not a goal in itself, but what we need to make it move to where we want. That is the job of the rest of **[Nav2](https://docs.nav2.org/)**: a *planner* computes a path to the goal over a costmap built from the map, and a *controller* sends the velocity commands to follow it, while AMCL tells them where the robot is. The demo can also launch these navigation servers:

```bash
ros2 launch particle_filter_demo demo_warehouse.launch.py nav2:=True
```

Their parameters are in `robotics_ws/src/particle_filter_demo/resources/nav2_params.yaml`. In RViz2, enable the displays `nav2_global_costmap` (the costmap used by the planner) and `nav2_plan` (the planned path). Before sending a goal, **press `p` in the window of `keyboard_control`** to pause it: otherwise it keeps publishing velocity commands, which fight those of Nav2.

1. Use the *2D Goal Pose* tool of the toolbar of RViz2 (Fig. 7) to send the robot to the other side of the warehouse, and include a **screenshot** of the path. Does the robot get there, even though its odometry drifts? Why? *Note: you can also send a goal from a terminal, e.g.:*

```bash
ros2 action send_goal /navigate_to_pose nav2_msgs/action/NavigateToPose "{pose: {header: {frame_id: map}, pose: {position: {x: 9.0, y: -7.0}, orientation: {w: 1.0}}}}"
```

2. While the robot navigates, *kidnap* AMCL with the *2D Pose Estimate* tool, as in Assignment 5. What happens with the navigation?

Write your answers below.

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

## <font color="orange"><b><i>AI Appendix</i></b></font>

### A.1 How I used AI
- **Model/tool:**
  <p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>Asistente de programación Devin (Claude Opus 5.5 + GPT-6 Luna).</i></p>
- **What I asked AI to help with (bullets):**
  <div style="margin: 4px 0px 6px 5px; color:#2DD4BF; font-style: italic"><ul><li>Guiarme paso a paso con los comandos de cada assignment, que yo ejecutaba en la máquina virtual y cuya salida y capturas le pasaba.</li><li>Instalar Nav2, clonar el repositorio y compilar el workspace <code>robotics_ws</code>.</li><li>Interpretar las salidas: topics de AMCL, árbol tf, número y pesos de las partículas, covarianza de <code>/amcl_pose</code> y errores frente a la pose real.</li><li>Detectar que AMCL se había perdido en el primer intento y en la localización global, comparando su pose con <code>/base_pose_ground_truth</code>.</li><li>Redactar las respuestas y hacer los cálculos (errores, desviaciones típicas, correlación, distancia de Mahalanobis).</li></ul></div>
- **Best prompt I used (paste):**
  <p style="margin: 4px 0px 6px 5px; color:#2DD4BF"></p>
- **What I kept vs. changed from the AI output:**
  <p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>Todos los valores del notebook salen de comandos ejecutados en mi simulación, no de la IA. El primer intento del Assignment 2 hubo que repetirlo: AMCL estaba a 13 m de la pose real porque había conducido demasiado rápido, y se relanzó la demo conduciendo despacio. La IA explicó al principio que las partículas se fusionaban «más tarde», pero en realidad lo hacían en uno o dos ciclos; para capturar la nube repartida hubo que llamar al servicio con el robot parado y moverlo un poco. La explicación de por qué salen 501 partículas y no 500 se comprobó en el código fuente de AMCL (<code>pf.c</code>).</i></p>

### A.2 AI review
Ask one generative AI to review your memo using this prompt:

```
Act as a technical reviewer of this mobile robotics lab notebook. Read my memo and return exactly 5 bullet points:

- two on theory (key ideas),
- two on practice (relevant ROS2 concepts and tools used, not particular commands or topics),
- one on how to improve the work (most critical aspect: limitations, parameter choices, robustness, possible mistakes).

Keep each bullet ≤15 words and make them specific to my work.
```

Paste its **5** items here:

*AI used for the review: the same AI assistant (Devin, Claude Opus 5.5 + GPT-6 Luna).*

- **(Theory 1)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>La odometría solo predice y deriva; el láser con el mapa acota el error de AMCL.</i></p>
- **(Theory 2)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>Sitios parecidos hacen la verosimilitud multimodal: la global y el secuestro fallaron por eso.</i></p>
- **(Practice 1)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>tf de un solo padre: AMCL publica map&rarr;odom y la odometría odom&rarr;base_link.</i></p>
- **(Practice 2)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>Nodos lifecycle, servicios y parámetros dinámicos para reinicializar y reconfigurar AMCL en marcha.</i></p>
- **(Improvement)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>Repetir el secuestro partiendo de AMCL bien localizado y cronometrar la recuperación.</i></p>

Finally, **do you agree with them?**, **would you have selected others?**
    <p style="margin: 4px 0px 6px 5px; color:#2DD4BF"><i>Estoy de acuerdo con los cinco. Los dos teóricos resumen lo que muestran los datos: 3.6 m de error de la odometría frente a 0.12 m de AMCL, y la convergencia a pasillos equivocados (21 y 24 m) cuando los scans de dos sitios se parecen. Los de práctica son las piezas de ROS 2 que más se han usado: el árbol tf, y los servicios y parámetros de un nodo en marcha. La mejora es justa: el secuestro se hizo con AMCL ya perdido y sin medir el tiempo, así que no mide bien la recuperación. Habría añadido un punto sobre la covarianza: con 20 partículas la elipse era mucho más pequeña que el error real (Mahalanobis &asymp; 4.9), así que no siempre se puede fiar uno de la incertidumbre que reporta el filtro.</i></p>